# Interactive VLM Hallucination Explorer

## Real-Time Attention & Logit Lens Visualization for LLaVA-1.5-7B

**Features:**
- **Layer Scrubber** — slide through 32 transformer layers and watch Yes/No belief evolve
- **Attention Heatmap Overlay** — spatial attention over the image, updating in real-time
- **VCD Noise Toggle** — inject diffusion noise and compare clean vs distorted attention
- **Logit Lens Readout** — decode top-5 tokens from high-attention image patches
- **Interactive Trajectory Plot** — Plotly scatter view of full layer-wise logit diff

**Built with:** ipywidgets, Plotly, Matplotlib, PIL, scipy
**Based on:** `image_attention_viz.py` + previous interpretability notebook

## 0. Setup & Imports

In [ ]:
import sys, subprocess, importlib

for pkg in ['transformers', 'accelerate', 'ipywidgets', 'plotly']:
    try:
        importlib.import_module(pkg.replace('-','_'))
        print(f'  OK  {pkg}')
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('Dependencies ready.')

In [ ]:
import os, sys, json, math
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from scipy.ndimage import gaussian_filter
from tqdm.auto import tqdm

# Interactive libraries
import ipywidgets as widgets
from ipywidgets import Layout, HBox, VBox, Tab, Output
from IPython.display import display, clear_output
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Paths ──
WORKSPACE_ROOT = Path.cwd().resolve()
VCD_ROOT   = WORKSPACE_ROOT / 'VCD'
EXP_ROOT   = VCD_ROOT / 'experiments'
LLAVA_ROOT = EXP_ROOT / 'llava'
DATA_DIR   = WORKSPACE_ROOT / 'data'

for p in [str(VCD_ROOT), str(EXP_ROOT), str(LLAVA_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f'Workspace: {WORKSPACE_ROOT}')
print(f'CUDA:      {torch.cuda.is_available()}')

In [ ]:
# ── Core utilities ──
# Token IDs
def get_yes_no_ids(tok):
    yes_id = tok.encode('Yes', add_special_tokens=False)[-1]
    no_id  = tok.encode('No',  add_special_tokens=False)[-1]
    return yes_id, no_id

def logit_diff(logits, yes_id, no_id):
    return logits[..., yes_id] - logits[..., no_id]

def project_logit_diff(hidden, lm_head, yes_id, no_id, ln=None):
    if ln is not None:
        hidden = ln(hidden)
    W = lm_head.weight if hasattr(lm_head, 'weight') else lm_head
    logits = hidden @ W.T
    return logits[..., yes_id] - logits[..., no_id]

# ── CLIP standard mean/std ──
CLIP_MEAN = np.array([0.48145466, 0.4578275, 0.40821073])
CLIP_STD  = np.array([0.26862954, 0.26130258, 0.27577711])

def denormalize_clip(tensor_3_h_w):
    """Convert CLIP-normalized [3,H,W] tensor to displayable [H,W,3] RGB."""
    arr = tensor_3_h_w.numpy() if hasattr(tensor_3_h_w, 'numpy') else tensor_3_h_w
    return (arr.transpose(1,2,0) * CLIP_STD + CLIP_MEAN).clip(0, 1)

print('Utilities ready.')

## 1. Load Model

Load LLaVA-1.5-7B using the VCD repo modules. We extract key architecture parameters needed for attention visualization.

In [ ]:
from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN
from llava.conversation import conv_templates
from llava.mm_utils import tokenizer_image_token, get_model_name_from_path
from llava.model.builder import load_pretrained_model

MODEL_PATH = "liuhaotian/llava-v1.5-7b"
MODEL_BASE = "lmsys/vicuna-7b-v1.5"

print(f"Loading {MODEL_PATH} ...")
tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path=MODEL_PATH,
    model_base=MODEL_BASE,
    model_name=get_model_name_from_path(MODEL_PATH),
    load_8bit=False,
    load_4bit=False,
)
model.eval()

# ── Architecture constants ──
vt         = model.get_vision_tower()
lm_head    = model.lm_head
final_ln   = model.model.norm
N_LAYERS   = len(model.model.layers)
HIDDEN_DIM = model.config.hidden_size
PATCH_PER_SIDE = 24       # 336 / 14
N_IMG_TOKENS   = 576      # 24 * 24

device = next(model.parameters()).device
dtype  = next(model.parameters()).dtype

yes_id, no_id = get_yes_no_ids(tokenizer)

print(f"  Layers: {N_LAYERS}, dim={HIDDEN_DIM}")
print(f"  Img tokens: {N_IMG_TOKENS} (grid {PATCH_PER_SIDE}x{PATCH_PER_SIDE})")
print(f"  Yes={yes_id}, No={no_id}")
print("Model ready.")

## 2. Load POPE Data

Load the benchmark and build UI-friendly data structures.